<a href="https://colab.research.google.com/github/DaisyXUUUUU/CIS-5450-Project/blob/main/notebooks/01_data_part_presentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 1 — Data: Acquisition, Sampling, Cleaning, Joins, and Split

**Owner:** Ziyue Xu (Daisy) · CIS 5450 Final Project · *Tipping Behavior in NYC Yellow Taxis: Patterns and Prediction Among Trips with Recorded Card Tips*

This is the **walk-through version** of the data part. It explains every data decision (problem → decision → reason) and
checks the result against the fixed analysis table. It does **not** change the data:

- The analysis table is **fixed at 796,989 trips**. By default this notebook only *reads* `trips_clean_v2.parquet`
  (= `trips_clean_v1.parquet` + 6 columns) from Google Drive and never writes any file.
- The sampling and cleaning code is shown in full. Steps that need the raw TLC files or the saved sample run only when
  `LIVE_REPLAY = True`; they first copy those files from Drive to the Colab disk (Drive is very slow for DuckDB), and they
  only **verify** (re-count, replay, compare); nothing is saved. With `LIVE_REPLAY = False` (default) the notebook runs
  in about a minute and shows the recorded numbers for those steps.
- Re-drawing the sample from raw files is behind `RERUN_PIPELINE = False`. Even when switched on, it only checks that the
  same 900,000 trips come back.
- Where a raw file is missing or live replay is off, the notebook shows the numbers recorded on the original run instead.

The notebooks that actually built the data are `01_data_pipeline.ipynb` (v1) and `01b_add_features_v2.ipynb` (v2); they are unchanged.

| Section | Topic |
| --- | --- |
| 0 | Research question, target, leakage rule, pipeline, setup |
| 1 | Data acquisition |
| 2 | Sampling with DuckDB (and the reservoir-sampling pitfall) |
| 3 | Data quality check before cleaning |
| 4 | Cleaning rules and row funnel |
| 5 | Joins: taxi zones and hourly weather |
| 6 | v2 features: rush hour, night, bill-based tip |
| 7 | Money-field check (`extra`, bill) |
| 8 | Time-based train / validation / test split |
| 9 | Output, data dictionary, summary |

Correlation checks and EDA (Yuzhi), class balance and leakage analysis for modeling (Yiwen), and encoding / scaling (modeling pipeline) are in teammates' notebooks.

## 0. Research question, target, and the leakage rule

**Question.** Among NYC yellow taxi trips with a positive recorded card tip, which time, location, trip, and weather
characteristics are associated with a high tip, and how well can a high tip be predicted?

**Target.** `high_tip = 1` if `tip_amount / fare_amount ≥ 25%` (proposal; 20% and 30% kept for sensitivity).
Section 6 adds the bill-based alternative `high_tip_bill` (tip as a share of everything paid before the tip).

**Model time point: the trip has ended, the rider has not paid yet.** So anything known at drop-off (time, place,
distance, duration, fare, surcharges, weather) may be a feature, but anything decided at payment may not:

> `tip_amount` and `total_amount` (which includes the tip), and every column computed from them (`tip_pct`, `high_tip*`,
> `tip_pct_bill`, `high_tip_bill*`), are **never** model features. They exist only to build the target.

**Pipeline.**

```
TLC monthly Parquet (Apr–Jun 2026, 11.76M trips)
  └─ DuckDB: card trips only, 300,000 per month, seed 5450  ──►  900,000 sampled trips
       └─ 8 cleaning rules, logged one by one                  ──►  796,989 trips
            └─ LEFT JOIN taxi zones (pickup + drop-off), LEFT JOIN hourly weather
                 └─ derived features, target, time split       ──►  trips_clean_v1.parquet (48 cols)
                      └─ + rush / night / bill-based tip        ──►  trips_clean_v2.parquet (54 cols)
```

**Run record.**

| Item | Value |
| --- | --- |
| Raw files downloaded | 2026-11-03 (TLC yellow Parquet Apr/May/Jun 2026, Taxi Zone Lookup CSV, Open-Meteo hourly weather) |
| Sample + cleaning run (v1) | 2026-11-03, `01_data_pipeline.ipynb`, repo commit `e28d7d4` |
| v2 columns added | 2026-11-16, `01b_add_features_v2.ipynb` (reads v1, no resampling) |
| Random seed | `SEED = 5450` (the only random step is the sample) |
| Software on that run | DuckDB 1.3.2, pandas 2.2.3 (Google Colab) |

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

try:
    import duckdb
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
    import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 80)
pd.set_option("display.width", 200)
print("duckdb", duckdb.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

### Switches and parameters

All design choices live in this one cell.

- `LIVE_REPLAY = False` (default): fast; Sections 1–4 show the numbers recorded on 2026-11-03 for the steps that need raw files.
- `LIVE_REPLAY = True`: copies the 3 raw TLC files (~200 MB) and the saved sample to the Colab disk once (a few minutes the
  first time in a session, instant afterwards), then re-counts and replays everything live and compares with the record.
- `RERUN_PIPELINE` stays `False`.

In [ ]:
# ---- switches -----------------------------------------------------------------------
LIVE_REPLAY = False           # True = copy raw files + saved sample from Drive to local disk and replay Sections 1-4 live.
RERUN_PIPELINE = False        # True = re-draw the 900k sample from the raw TLC files and check it matches the saved one.
                              #        Never overwrites anything. Needs LIVE_REPLAY = True and DuckDB 1.3.2.
DEMO_RESERVOIR_PITFALL = False  # True = re-run the first (rejected) reservoir sample on May to show the time bias.

# ---- study design (fixed in the proposal) --------------------------------------------
SEED = 5450
MONTHS = ["2026-04", "2026-05", "2026-06"]
N_PER_MONTH = 300_000
STUDY_START, STUDY_END = pd.Timestamp("2026-04-01"), pd.Timestamp("2026-07-01")   # [start, end)
TRAIN_END = pd.Timestamp("2026-05-16")   # train: Apr 1 – May 15
VAL_END   = pd.Timestamp("2026-06-01")   # val:   May 16 – May 31; test: June
HIGH_TIP_THRESHOLD = 0.25
RAIN_MM = 0.1
JFK, LGA, EWR = 132, 138, 1
UNKNOWN_ZONES = [264, 265]

# ---- paths (Google Drive) -------------------------------------------------------------
DATA_DIR = Path(os.environ.get("CIS5450_DATA_DIR", "/content/drive/MyDrive/CIS5450_project/data"))
RAW_DIR, CLEAN_DIR = DATA_DIR / "raw", DATA_DIR / "clean"
V1_PATH, V2_PATH = CLEAN_DIR / "trips_clean_v1.parquet", CLEAN_DIR / "trips_clean_v2.parquet"
SAMPLE_PATH  = RAW_DIR / "sample_card_trips.parquet"     # the 900,000-trip sample saved on 2026-11-03
ZONES_PATH   = RAW_DIR / "taxi_zone_lookup.csv"
WEATHER_PATH = RAW_DIR / "weather_hourly_nyc.csv"
RAW_PATHS    = {m: RAW_DIR / f"yellow_tripdata_{m}.parquet" for m in MONTHS}

# ---- fixed results that every run must reproduce ---------------------------------------
EXPECTED_ROWS = 796_989
EXPECTED_SPLIT_ROWS = {"train": 395_144, "val": 127_349, "test": 274_496}
EXPECTED_SPLIT_HIGH_TIP = {"train": 0.633, "val": 0.624, "test": 0.630}   # rounded to 3 decimals

HAVE = {
    "v2 parquet": V2_PATH.exists(),
    "v1 parquet": V1_PATH.exists(),
    "saved 900k sample": SAMPLE_PATH.exists(),
    "raw TLC files (3 months)": all(p.exists() for p in RAW_PATHS.values()),
    "zone lookup": ZONES_PATH.exists(),
    "weather csv": WEATHER_PATH.exists(),
}
display(pd.Series(HAVE, name="found on Drive").to_frame())
assert HAVE["v2 parquet"] or HAVE["v1 parquet"], f"Neither {V2_PATH} nor {V1_PATH} exists. Check the Drive path."

### Local copies for live replay

DuckDB reads Parquet with many small random reads, which are extremely slow on the Google Drive mount (a single count
can take more than 10 minutes). So, only when `LIVE_REPLAY = True`, the files are copied once to the Colab disk and all
queries run on the copies. The copies are deleted when the Colab session ends; the Drive files are only read.

In [ ]:
import shutil, tempfile, time
LOCAL_DIR = Path("/content/cis5450_local") if IN_COLAB else Path(tempfile.gettempdir()) / "cis5450_local"
RAW_LOCAL = {m: LOCAL_DIR / p.name for m, p in RAW_PATHS.items()}
SAMPLE_LOCAL = LOCAL_DIR / SAMPLE_PATH.name

def local_copy(src, dst):
    if dst.exists() and dst.stat().st_size == src.stat().st_size:
        print(f"[local copy exists] {dst.name}")
        return
    t0 = time.time()
    shutil.copyfile(src, dst)
    print(f"[copied] {src.name}  {dst.stat().st_size / 1e6:.0f} MB in {time.time() - t0:.0f} s")

USE_RAW = LIVE_REPLAY and HAVE["raw TLC files (3 months)"]
USE_SAMPLE = LIVE_REPLAY and HAVE["saved 900k sample"]
if LIVE_REPLAY:
    LOCAL_DIR.mkdir(parents=True, exist_ok=True)
    if USE_SAMPLE:
        local_copy(SAMPLE_PATH, SAMPLE_LOCAL)
    if USE_RAW:
        for m in MONTHS:
            local_copy(RAW_PATHS[m], RAW_LOCAL[m])
print(f"LIVE_REPLAY={LIVE_REPLAY} | live raw-file steps: {USE_RAW} | live sample steps: {USE_SAMPLE}")

### Load the fixed analysis table (read only)

We load v2 if it exists (it contains all 48 v1 columns unchanged plus 6 new ones); otherwise v1, and Section 6 rebuilds the six columns.
A fingerprint of the loaded table is taken now and checked again at the very end, to show this notebook changed nothing.

In [ ]:
LOADED_FROM = V2_PATH if HAVE["v2 parquet"] else V1_PATH
df = pd.read_parquet(LOADED_FROM)
LOADED_COLS = list(df.columns)
LOADED_FINGERPRINT = pd.util.hash_pandas_object(df, index=True).sum()

print(f"Loaded {LOADED_FROM.name}: {df.shape[0]:,} rows x {df.shape[1]} columns")
print("Pickup range:", df["tpep_pickup_datetime"].min(), "to", df["tpep_pickup_datetime"].max())
assert len(df) == EXPECTED_ROWS, f"expected {EXPECTED_ROWS:,} rows, got {len(df):,}"
assert df.shape[1] in (48, 54), df.shape
df.head(3)

In [ ]:
con = duckdb.connect()

## 1. Data acquisition

**Sources.**

| Data | Source | Grain | Used for |
| --- | --- | --- | --- |
| Yellow taxi trip records, April–June 2026 | [NYC TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page), one Parquet file per month | one row per trip | everything |
| Taxi Zone Lookup Table | same page (`taxi_zone_lookup.csv`) | one row per zone (265) | borough / zone names for pickup and drop-off |
| Hourly weather, Manhattan | [Open-Meteo Historical Weather API](https://open-meteo.com/en/docs/historical-weather-api) (`temperature_2m`, `precipitation`, New York time) | one row per hour (2,184) | temperature, rain flag |

**Why these months.** April–June 2026 are three recent consecutive months, all after the CBD congestion fee began
(2025-01-05), so the fare and surcharge rules are the same across train, validation, and test. Three months give ~11.8
million trips, far above the course minimum.

**Decision.** Files were downloaded once (2026-11-03) to Google Drive and reused, so the analysis does not depend on the
TLC site staying the same (TLC sometimes re-publishes files).

In [ ]:
TLC_BASE = "https://d37ci6vzurychx.cloudfront.net"
SOURCES = {f"yellow_tripdata_{m}.parquet": f"{TLC_BASE}/trip-data/yellow_tripdata_{m}.parquet" for m in MONTHS}
SOURCES["taxi_zone_lookup.csv"] = f"{TLC_BASE}/misc/taxi_zone_lookup.csv"

def download(url, dest):
    """Download once; an existing file is never overwritten."""
    import requests
    if dest.exists() and dest.stat().st_size > 0:
        print(f"[cached] {dest.name}")
        return
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        tmp = dest.with_suffix(dest.suffix + ".part")
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
        tmp.rename(dest)
    print(f"[download] {dest.name}")

if RERUN_PIPELINE:
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    for name, url in SOURCES.items():
        download(url, RAW_DIR / name)
else:
    for name, url in SOURCES.items():
        print(f"{name:34s} <- {url}")

**Size of the raw data.** Counted with DuckDB directly on the Parquet files (nothing is loaded into pandas).
Row and column counts come from the Parquet footer; the card count scans one column. With live replay off, the counts recorded on 2026-11-03 are shown.

In [ ]:
RECORDED_RAW = pd.DataFrame({
    "month":      MONTHS,
    "rows":       [3_831_240, 4_090_836, 3_837_248],
    "columns":    [20, 20, 21],
    "card_trips": [2_635_749, 2_727_585, 2_432_871],
})
RECORDED_RAW["card_share"] = RECORDED_RAW["card_trips"] / RECORDED_RAW["rows"]

if USE_RAW:
    import pyarrow.parquet as pq
    rows = []
    for m in MONTHS:
        p = str(RAW_LOCAL[m])
        meta = pq.ParquetFile(p).metadata                      # row and column counts from the file footer, no scan
        n_rows, n_cols = meta.num_rows, meta.num_columns
        n_card = con.execute(f"SELECT count(*) FROM read_parquet('{p}') WHERE payment_type = 1").fetchone()[0]
        rows.append({"month": m, "rows": n_rows, "columns": n_cols, "card_trips": n_card, "card_share": n_card / n_rows})
    raw_summary = pd.DataFrame(rows)
    same = raw_summary[["rows", "columns", "card_trips"]].equals(RECORDED_RAW[["rows", "columns", "card_trips"]])
    print("Live count from raw files. Matches the 2026-11-03 record:", same)
else:
    raw_summary = RECORDED_RAW
    print("Live replay off (or raw files not on Drive): showing the counts recorded on 2026-11-03.")

display(raw_summary.style.format({"rows": "{:,}", "card_trips": "{:,}", "card_share": "{:.1%}"}))
print(f"Total raw trips: {raw_summary['rows'].sum():,} | card trips: {raw_summary['card_trips'].sum():,}")

In [ ]:
# Raw schema (June). June has one extra column, `request_source`, that is not in the TLC data dictionary.
if USE_RAW:
    display(con.execute(f"DESCRIBE SELECT * FROM read_parquet('{RAW_LOCAL['2026-06']}')").df()[["column_name", "column_type"]])
else:
    print("Live replay off. The 21 raw columns are: VendorID, tpep_pickup_datetime, tpep_dropoff_datetime, "
          "passenger_count, trip_distance, RatecodeID, store_and_fwd_flag, PULocationID, DOLocationID, payment_type, "
          "fare_amount, extra, mta_tax, tip_amount, tolls_amount, improvement_surcharge, total_amount, "
          "congestion_surcharge, Airport_fee, cbd_congestion_fee, request_source (June only).")

**Conclusion.** 11,759,324 raw trips; 63–69% are credit-card trips. We only use card trips, because TLC records tips
only for card payments (a cash tip is always stored as 0).

## 2. Sampling with DuckDB

**Problem.** 7.8 million card trips is more than we need and slow to iterate on in Colab (and some models, e.g. the
FT-Transformer, are expensive). We want a smaller sample that is still representative of every day and hour.

**Decision.** From each month keep exactly **300,000 card trips** (`payment_type = 1`), drawn uniformly at random with
`SEED = 5450` → 900,000 trips. DuckDB queries the Parquet files directly, so the raw rows never enter pandas.

How: each card trip gets a pseudo-random key `hash(file_row_number, SEED)`; we keep the 300,000 trips with the smallest keys.
The key does not depend on the trip's content or its position in the file, so every card trip has the same chance, and the
same seed returns the same trips.

**Pitfall we hit first (and why the check below exists).** The first version used
`SELECT * FROM (... WHERE payment_type = 1) USING SAMPLE reservoir(300000 ROWS) REPEATABLE (5450)`.
TLC files are roughly sorted by time, and that sample came out concentrated in the first half of each month: after cleaning,
the May 16–31 **validation fold had only 10,836 trips** (it should have ~127,000). Nothing crashed; we only noticed from the
fold sizes. That run was discarded. We switched to hash sampling and added an automatic check that compares the share of
trips on each day in the sample with the full month.

**Equal 300k per month** (rather than proportional): months differ by under 7% in card trips, and equal counts keep the
months balanced. Day-of-month shares *within* a month stay proportional, which is what the check verifies.

In [ ]:
SAMPLE_SQL = '''
    SELECT * EXCLUDE (file_row_number)
    FROM read_parquet('{path}', file_row_number = true)
    WHERE payment_type = 1                         -- card trips only
    ORDER BY hash(file_row_number, {seed})         -- seeded pseudo-random key
    LIMIT {n}
'''

def draw_sample(con, n=N_PER_MONTH, seed=SEED):
    parts = []
    for m in MONTHS:
        part = con.execute(SAMPLE_SQL.format(path=RAW_LOCAL[m], seed=seed, n=n)).df()
        part["source_month"] = m
        parts.append(part)
    return pd.concat(parts, ignore_index=True)

KEY_COLS = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", "DOLocationID",
            "trip_distance", "fare_amount", "tip_amount", "total_amount"]

def key_frame(d):
    """Order-free fingerprint of a set of trips, used to compare two tables row by row."""
    k = d[KEY_COLS].copy()
    for c in KEY_COLS[:2]:
        k[c] = pd.to_datetime(k[c]).astype("datetime64[ns]")
    for c in KEY_COLS[2:]:
        k[c] = k[c].astype("float64")
    return k.sort_values(KEY_COLS).reset_index(drop=True)

print(SAMPLE_SQL.format(path="<month file>", seed=SEED, n=N_PER_MONTH))

In [ ]:
# Load the sample that was drawn on 2026-11-03 (read only). Sections 2-4 replay on it if it exists.
sample = None
if USE_SAMPLE:
    sample = pd.read_parquet(SAMPLE_LOCAL)
    for c in ["tpep_pickup_datetime", "tpep_dropoff_datetime"]:
        sample[c] = pd.to_datetime(sample[c])
    if "source_month" not in sample:
        sample["source_month"] = sample["tpep_pickup_datetime"].dt.strftime("%Y-%m")
    print(f"Saved sample: {len(sample):,} rows x {sample.shape[1]} columns")
    print(sample["source_month"].value_counts().sort_index().to_string())
    print("Exactly 300,000 per month:", sample["source_month"].value_counts().eq(N_PER_MONTH).all() and len(sample) == 900_000)
else:
    print("Live replay off (or saved sample not on Drive): the recorded result is 900,000 trips (300,000 per month).")

if RERUN_PIPELINE:
    assert USE_RAW, "RERUN_PIPELINE needs LIVE_REPLAY = True and the raw files on Drive"
    if duckdb.__version__ != "1.3.2":
        print(f"WARNING: DuckDB {duckdb.__version__}; the sample was drawn with 1.3.2 and hash() may differ across versions.")
    redraw = draw_sample(con)               # kept in memory only, never saved
    print(f"Re-drawn sample: {len(redraw):,} rows")
    if sample is not None:
        print("Re-drawn sample is the same set of trips as the saved one:", key_frame(redraw).equals(key_frame(sample)))

### Check: is the sample spread evenly over each month?

For each month, compare the share of card trips on each day in the full file with the share in the sample.
With 300,000 trips per month an unbiased sample stays within ~0.1–0.15 percentage points on every day; the original
pipeline stops if any day is off by more than 0.3 points (the reservoir sample was off by several points).

In [ ]:
RECORDED_DAILY_CHECK = pd.DataFrame({
    "month": MONTHS,
    "full_month_share_days_1_15": [0.4776, 0.5112, 0.5260],
    "sample_share_days_1_15":     [0.4785, 0.5121, 0.5268],
    "max_daily_gap_pct_points":   [0.0541, 0.0626, 0.0493],
})

if USE_RAW and sample is not None:
    files = ", ".join(repr(str(RAW_LOCAL[m])) for m in MONTHS)
    pop_daily = con.execute(f'''
        SELECT CAST(tpep_pickup_datetime AS DATE) AS day, count(*) AS n
        FROM read_parquet([{files}], union_by_name = true)
        WHERE payment_type = 1
        GROUP BY ALL
    ''').df()
    pop_daily["day"] = pd.to_datetime(pop_daily["day"])
    samp_daily = (sample.assign(day=sample["tpep_pickup_datetime"].dt.normalize())
                        .groupby(["source_month", "day"]).size().rename("n_sample").reset_index())
    rows = []
    for m in MONTHS:
        p = pop_daily[pop_daily["day"].dt.strftime("%Y-%m") == m].set_index("day")["n"]
        q = samp_daily[(samp_daily["source_month"] == m) & (samp_daily["day"].dt.strftime("%Y-%m") == m)].set_index("day")["n_sample"]
        share = pd.concat([p / p.sum(), q / q.sum()], axis=1, keys=["full_month", "sample"]).fillna(0)
        first_half = share.index.day <= 15
        rows.append({"month": m,
                     "full_month_share_days_1_15": share.loc[first_half, "full_month"].sum(),
                     "sample_share_days_1_15": share.loc[first_half, "sample"].sum(),
                     "max_daily_gap_pct_points": (share["full_month"] - share["sample"]).abs().max() * 100})
    daily_check = pd.DataFrame(rows)
    print("Live check on the saved sample vs the raw files:")
else:
    daily_check = RECORDED_DAILY_CHECK
    print("Live replay off (or files missing): showing the check recorded on 2026-11-03.")

display(daily_check.round(4))
print("Every day within 0.3 percentage points:", bool((daily_check["max_daily_gap_pct_points"] < 0.3).all()))

In [ ]:
# Optional demo of the rejected method (off by default; it is not used anywhere).
if DEMO_RESERVOIR_PITFALL and USE_RAW:
    con.execute("SET threads = 1")
    bad = con.execute(f'''
        SELECT tpep_pickup_datetime FROM (
            SELECT * FROM read_parquet('{RAW_LOCAL["2026-05"]}') WHERE payment_type = 1
        ) USING SAMPLE reservoir({N_PER_MONTH} ROWS) REPEATABLE ({SEED})
    ''').df()
    con.execute("RESET threads")
    late = (pd.to_datetime(bad["tpep_pickup_datetime"]).dt.day >= 16).mean()
    print(f"Reservoir sample, May: share of trips on May 16-31 = {late:.1%} (full month: about 49%)")
else:
    print("Demo off. Recorded result of the rejected reservoir sample: validation fold (May 16-31) = 10,836 trips after cleaning.")

**Conclusion.** 900,000 card trips, exactly 300,000 per month, and each day's share in the sample is within 0.07
percentage points of the full month. The sample is saved on Drive and is the only random step in the project, so every
later number is reproducible without re-sampling.

## 3. Data quality check before cleaning

**Goal.** Look at the 900,000 sampled trips *before* any rule is applied: column types, missing values, value ranges, and
how many trips show each kind of problem. Each problem below is counted on the whole sample independently (one trip can
have several problems), so these counts overlap; Section 4 removes them in order without double counting.

In [ ]:
if sample is not None:
    q = sample
    dtypes_missing = pd.DataFrame({"dtype": q.dtypes.astype(str),
                                   "missing": q.isna().sum(),
                                   "missing_pct": (q.isna().mean() * 100).round(2)})
    display(dtypes_missing)
else:
    print("Live replay off (or saved sample not on Drive), so the pre-cleaning profile is not shown; "
          "see the recorded funnel in Section 4.")

In [ ]:
if sample is not None:
    q = sample
    q_duration = (q["tpep_dropoff_datetime"] - q["tpep_pickup_datetime"]).dt.total_seconds() / 60
    num = q[["trip_distance", "fare_amount", "tip_amount", "total_amount", "extra", "tolls_amount", "passenger_count"]].assign(duration_min=q_duration)
    display(num.describe(percentiles=[0.01, 0.5, 0.99]).T.round(2))

In [ ]:
if sample is not None:
    q = sample
    with np.errstate(divide="ignore", invalid="ignore"):
        q_tip_ratio = q["tip_amount"] / q["fare_amount"].where(q["fare_amount"] > 0)
    issues = pd.Series({
        "tip_amount = 0 (no card tip recorded)":    (q["tip_amount"] == 0).sum(),
        "tip_amount < 0":                            (q["tip_amount"] < 0).sum(),
        "fare_amount <= 0":                          (q["fare_amount"] <= 0).sum(),
        "pickup outside Apr 1 - Jun 30, 2026":       (~q["tpep_pickup_datetime"].between(STUDY_START, STUDY_END, inclusive="left")).sum(),
        "duration < 1 min":                          (q_duration < 1).sum(),
        "duration > 180 min":                        (q_duration > 180).sum(),
        "trip_distance = 0":                         (q["trip_distance"] <= 0).sum(),
        "trip_distance > 100 mi":                    (q["trip_distance"] > 100).sum(),
        "pickup or drop-off zone 264/265 (unknown)": (q["PULocationID"].isin(UNKNOWN_ZONES) | q["DOLocationID"].isin(UNKNOWN_ZONES)).sum(),
        "RatecodeID = 99 (unknown)":                 (q["RatecodeID"] == 99).sum(),
        "tip > fare (tip/fare > 100%)":              (q_tip_ratio > 1).sum(),
        "passenger_count = 0":                       (q["passenger_count"] == 0).sum(),
        "request_source not empty":                  q["request_source"].notna().sum() if "request_source" in q else 0,
    }, name="trips").to_frame()
    issues["pct_of_sample"] = (issues["trips"] / len(q) * 100).round(2)
    display(issues)

**What we found and what we do about it.**

| Issue | Decision | Reason |
| --- | --- | --- |
| `tip_amount = 0` (~8%) | drop | Study population is trips **with a recorded card tip**. A zero card tip cannot tell "did not tip" from "tipped in cash", so it is not a reliable "no tip" label. |
| `fare_amount ≤ 0` | drop | Refunds / voids / errors; tip ÷ fare is undefined. |
| pickup outside April–June | drop | TLC files contain a few trips with wrong timestamps. |
| duration < 1 min or > 180 min | drop | Under a minute is a meter start/stop error; over three hours is not a single street-hail trip. |
| distance = 0 or > 100 mi | drop | Zero is a meter / GPS error; > 100 mi is outside NYC yellow-cab service. |
| zone 264 / 265 | drop | "Unknown" / "Outside of NYC": cannot be mapped to a borough, so location features would be meaningless. |
| `RatecodeID = 99` | drop | Unknown rate code (0 trips remain after the rules above, kept for completeness). |
| tip > fare | drop | Mostly keying errors (e.g. tip typed in the fare field); a handful of extreme ratios would dominate every ratio summary. |
| `passenger_count = 0` | **keep the trip**, set count to missing | Only the count is wrong, the trip is fine. Imputation happens in the modeling pipeline on train only. |
| `request_source` | drop the **column** | Not in the TLC data dictionary, absent in April–May, and empty for every sampled card trip in June. |
| very slow trips (short distance, long time) | **keep** | Could be real traffic; see Section 4. |

All thresholds are fixed constants from domain knowledge, not quantiles of the data, so June (test) data does not influence any rule.

## 4. Cleaning: rules and row funnel

Rules are applied one at a time in this order, and each step logs how many trips it removed. The code below is the exact
code from `01_data_pipeline.ipynb`. If the saved sample is on Drive it is replayed here (in memory only) and compared with
the fixed table; otherwise the recorded funnel is shown.

In [ ]:
RULES = [
    ("1. tip_amount > 0",                     lambda d: d["tip_amount"] > 0),
    ("2. fare_amount > 0",                    lambda d: d["fare_amount"] > 0),
    ("3. pickup within Apr 1 - Jun 30, 2026", lambda d: d["tpep_pickup_datetime"].between(STUDY_START, STUDY_END, inclusive="left")),
    ("4. 1 <= duration <= 180 min",           lambda d: d["duration_min"].between(1, 180)),
    ("5. 0 < trip_distance <= 100 mi",        lambda d: (d["trip_distance"] > 0) & (d["trip_distance"] <= 100)),
    ("6. PU/DO zone not 264/265",             lambda d: ~d["PULocationID"].isin(UNKNOWN_ZONES) & ~d["DOLocationID"].isin(UNKNOWN_ZONES)),
    ("7. RatecodeID != 99",                   lambda d: d["RatecodeID"].ne(99) | d["RatecodeID"].isna()),
    ("8. tip_pct <= 100%",                    lambda d: d["tip_amount"] / d["fare_amount"] <= 1.0),
]

def run_funnel(d):
    d = d.drop(columns=["request_source"], errors="ignore").copy()
    d["duration_min"] = (d["tpep_dropoff_datetime"] - d["tpep_pickup_datetime"]).dt.total_seconds() / 60
    log = [{"step": "0. sampled card trips", "rows_after": len(d), "removed": 0}]
    for name, rule in RULES:
        before = len(d)
        d = d[rule(d).fillna(False).astype(bool)]
        log.append({"step": name, "rows_after": len(d), "removed": before - len(d)})
    d = d.reset_index(drop=True)
    d.loc[d["passenger_count"] == 0, "passenger_count"] = np.nan     # invalid count -> missing (trip kept)
    return d, pd.DataFrame(log)

RECORDED_FUNNEL = pd.DataFrame({
    "step": ["0. sampled card trips"] + [name for name, _ in RULES],
    "rows_after": [900_000, 825_235, 825_233, 825_232, 804_723, 802_836, 798_194, 798_194, 796_989],
    "removed":    [0, 74_765, 2, 1, 20_509, 1_887, 4_642, 0, 1_205],
})

In [ ]:
if sample is not None:
    replayed, funnel = run_funnel(sample)
    funnel_matches = funnel[["rows_after", "removed"]].equals(RECORDED_FUNNEL[["rows_after", "removed"]])
    rows_match = key_frame(replayed).equals(key_frame(df))
    print("Replayed the 8 rules on the saved sample.")
    print("  funnel matches the 2026-11-03 record:", funnel_matches)
    print("  replayed trips are exactly the trips in the fixed table:", rows_match)
    if not (funnel_matches and rows_match):
        print("  WARNING: the saved sample on Drive does not reproduce the fixed table. Showing the recorded funnel.")
        funnel = RECORDED_FUNNEL.copy()
else:
    funnel = RECORDED_FUNNEL.copy()
    print("Live replay off (or saved sample not on Drive): showing the funnel recorded on 2026-11-03.")

funnel["pct_removed_of_sample"] = (funnel["removed"] / funnel.loc[0, "rows_after"] * 100).round(2)
funnel["pct_kept_so_far"] = (funnel["rows_after"] / funnel.loc[0, "rows_after"] * 100).round(2)
display(funnel.style.format({"rows_after": "{:,}", "removed": "{:,}"}))
assert funnel["rows_after"].iloc[-1] == EXPECTED_ROWS == len(df)

Reading the funnel: after rules 1–2 (positive tip and fare) **825,233** trips remain (the proposal reported 825,072 from
the first, discarded sample; the small gap is from re-drawing with the unbiased method). Duration is the only other rule
that removes more than 1%. Final: **796,989 trips (88.6% of the sample)**.

### Things we deliberately did *not* remove

**Slow trips (short distance, long duration).** A 1-mile trip that takes 40 minutes looks odd, but in Manhattan rush hour
it can be real traffic. We have no ground truth to tell a meter error from a traffic jam, and dropping them would remove
exactly the congested trips that matter for the rush-hour question. So only the 1–180 minute rule applies.

In [ ]:
slow = df["avg_speed_mph"] < 3
print(f"Trips slower than 3 mph: {slow.sum():,} ({slow.mean():.2%}) -- kept")
print(f"  share in weekday 16-19h:           {df.loc[slow, 'is_rush_hour'].mean():.1%}  (all trips: {df['is_rush_hour'].mean():.1%})"
      if "is_rush_hour" in df else "")
print(f"  share picked up in Manhattan:      {(df.loc[slow, 'PU_Borough'] == 'Manhattan').mean():.1%}  (all trips: {(df['PU_Borough'] == 'Manhattan').mean():.1%})")
print(f"  median distance / duration:        {df.loc[slow, 'trip_distance'].median():.2f} mi / {df.loc[slow, 'duration_min'].median():.1f} min")

n_fast = int((df["avg_speed_mph"] > 80).sum())
print(f"Trips faster than 80 mph: {n_fast} -- kept here; any cap is chosen in feature engineering on the train split only")
n_pass_missing = int(df["passenger_count"].isna().sum())
print(f"passenger_count missing (was 0): {n_pass_missing:,} ({n_pass_missing / len(df):.2%}) -- imputed later on train only")
print("Other missing values in the fixed table:", int(df.drop(columns=["passenger_count", "tip_pct_bill"], errors="ignore").isna().sum().sum()))

**Conclusion.** 796,989 clean trips, every removal explained and counted. No values were imputed, capped, or scaled here:
those steps learn from the data and therefore belong in the modeling pipeline, fitted on the training split only.

## 5. Joins: taxi zones and hourly weather

**Zone join.** `PULocationID` and `DOLocationID` are each LEFT JOINed to the zone lookup (two joins, one per trip end),
adding borough, zone name, and service zone. LEFT JOIN so that a missing match shows up as a null we can count, instead of
silently dropping the trip.

**Weather join — alignment rule (no future information).** Open-Meteo labels hour *T* with the precipitation that fell
from *T − 1h* to *T*. We floor each pickup time to the hour (`pickup_hour_ts`) and join on it, which picks the most recent
hour that had ended by pickup. Rounding to the nearest hour would sometimes use an hour that had not finished yet.
TLC timestamps are New York local time; the weather was requested in `America/New_York`; April–June 2026 has no
daylight-saving change (it was March 8), so every local hour is unique.

`is_rain = precip_mm ≥ 0.1` (smallest amount the API reports as non-zero rain).

Below, the joins are **re-run** with DuckDB on the fixed table's IDs and compared with the stored columns.

In [ ]:
ids = df[["PULocationID", "DOLocationID", "pickup_hour_ts"]].copy()
ids["row_id"] = np.arange(len(ids))
con.register("trip_ids", ids)

if HAVE["zone lookup"]:
    zones = pd.read_csv(ZONES_PATH)
    con.register("zones", zones)
    rejoined = con.execute('''
        SELECT t.row_id,
               pz.Borough AS PU_Borough, pz.Zone AS PU_Zone, pz.service_zone AS PU_service_zone,
               dz.Borough AS DO_Borough, dz.Zone AS DO_Zone, dz.service_zone AS DO_service_zone
        FROM trip_ids AS t
        LEFT JOIN zones AS pz ON t.PULocationID = pz.LocationID
        LEFT JOIN zones AS dz ON t.DOLocationID = dz.LocationID
        ORDER BY t.row_id
    ''').df()
    assert len(rejoined) == len(df), "zone join changed the row count"
    zone_cols = ["PU_Borough", "PU_Zone", "PU_service_zone", "DO_Borough", "DO_Zone", "DO_service_zone"]
    same = all((rejoined[c].fillna("<NA>").to_numpy() == df[c].fillna("<NA>").to_numpy()).all() for c in zone_cols)
    print(f"Zone lookup: {len(zones)} zones. Re-run join reproduces the stored zone columns: {same}")
else:
    print("Zone lookup not on Drive; checking the stored columns only.")

print("Trips with no pickup borough:", int(df["PU_Borough"].isna().sum()), "| no drop-off borough:", int(df["DO_Borough"].isna().sum()))
assert df["PU_Borough"].notna().all() and df["DO_Borough"].notna().all()
display(pd.crosstab(df["PU_Borough"], df["DO_Borough"], margins=True))

In [ ]:
if HAVE["weather csv"]:
    weather = pd.read_csv(WEATHER_PATH, parse_dates=["time"]).rename(
        columns={"time": "weather_hour", "temperature_2m": "temp_c", "precipitation": "precip_mm"})
    expected_hours = (STUDY_END - STUDY_START) // pd.Timedelta(hours=1)
    print(f"Weather hours: {len(weather):,} (expected {expected_hours:,}), duplicates: {weather['weather_hour'].duplicated().sum()}, "
          f"missing values: {int(weather[['temp_c', 'precip_mm']].isna().sum().sum())}")
    print(f"Rainy hours (>= {RAIN_MM} mm): {(weather['precip_mm'] >= RAIN_MM).sum()} of {len(weather)}")
    con.register("weather", weather)
    rejoined_w = con.execute('''
        SELECT t.row_id, w.temp_c, w.precip_mm
        FROM trip_ids AS t
        LEFT JOIN weather AS w ON t.pickup_hour_ts = w.weather_hour
        ORDER BY t.row_id
    ''').df()
    assert len(rejoined_w) == len(df), "weather join changed the row count"
    same = (np.allclose(rejoined_w["temp_c"], df["temp_c"], equal_nan=True)
            and np.allclose(rejoined_w["precip_mm"], df["precip_mm"], equal_nan=True))
    print("Re-run weather join reproduces the stored temp_c and precip_mm:", same)
else:
    print("Weather csv not on Drive; checking the stored columns only.")

# the alignment rule: the joined hour is the pickup time floored to the hour, never later than pickup
assert (df["pickup_hour_ts"] == df["tpep_pickup_datetime"].dt.floor("h")).all()
assert (df["pickup_hour_ts"] <= df["tpep_pickup_datetime"]).all()
coverage = df["temp_c"].notna().mean()
print(f"Weather coverage: {coverage:.2%} | trips in a rainy hour: {int(df['is_rain'].sum()):,} ({df['is_rain'].mean():.1%})")
assert coverage == 1.0

**Conclusion.** Both joins keep the row count (796,989) and match **100%** of trips: every trip has a pickup and drop-off
borough and an hour of weather. 16.2% of trips started in a rainy hour (319 of 2,184 hours had ≥ 0.1 mm).
91% of pickups are in Manhattan and most of the rest in Queens (the airports), so outer-borough results will be less precise.

## 6. v2 features: rush hour, night, and bill-based tip

v2 adds six columns to v1 **without touching any row or any existing value** (`01b_add_features_v2.ipynb`). Here the six
columns are recomputed from v1 columns and compared with the stored ones.

| Column | Definition | Why |
| --- | --- | --- |
| `is_rush_hour` | Mon–Fri, pickup 16:00–19:59 | TLC's $2.50 rush-hour surcharge window (keyed on pickup time). Feature. |
| `is_night` | pickup 20:00–05:59 | TLC's $1 overnight surcharge window. Feature. |
| `tip_pct_bill` | `tip_amount / (total_amount − tip_amount)` | Tip as a share of everything the rider paid before the tip, i.e. what the payment screen's % buttons are applied to. Outcome, **not a feature**. |
| `high_tip_bill` | tip > `round(0.20 × bill, 2) + $0.015` | Tip above the 20% preset by more than a cent (rounding to the cent would otherwise push preset tips to either side of 0.20). Alternative target, **not a feature**. |
| `high_tip_bill_25`, `_30` | same rule at 25% / 30% | Sensitivity checks, **not features**. |

**Holidays are not excluded from `is_rush_hour`.** The TLC rule text says the rush surcharge does not apply on legal
holidays, but on the two federal holidays in our window (Memorial Day, Mon May 25; Juneteenth, Fri June 19) about 90% of
Vendor 2 trips between 16:00 and 19:59 were charged `extra = 2.50`, close to the ~98% of an ordinary weekday. The flag
follows what riders were actually charged (check printed below).

Why the bill-based denominator was added at all is part of the EDA (Yuzhi): the hour-of-day pattern in tip ÷ fare is
largely driven by the surcharges in the denominator.

In [ ]:
pu = df["tpep_pickup_datetime"]
hour, dow = pu.dt.hour, pu.dt.dayofweek
bill = (df["total_amount"] - df["tip_amount"]).round(2)

v2_cols = pd.DataFrame(index=df.index)
v2_cols["is_rush_hour"] = ((dow <= 4) & hour.between(16, 19)).astype("int8")
v2_cols["is_night"] = ((hour >= 20) | (hour <= 5)).astype("int8")
v2_cols["tip_pct_bill"] = np.where(bill > 0, df["tip_amount"] / bill, np.nan)
v2_cols["high_tip_bill"] = (df["tip_amount"] > (0.20 * bill).round(2) + 0.015).astype("int8")
for t in (0.25, 0.30):
    v2_cols[f"high_tip_bill_{int(t * 100)}"] = (df["tip_amount"] > (t * bill).round(2) + 0.015).astype("int8")
for c in ["high_tip_bill", "high_tip_bill_25", "high_tip_bill_30"]:
    v2_cols.loc[bill <= 0, c] = 0
V2_NEW = list(v2_cols.columns)

if all(c in df for c in V2_NEW):
    for c in V2_NEW:
        pd.testing.assert_series_equal(df[c], v2_cols[c], check_names=False, check_dtype=False)
    print("All 6 v2 columns recomputed from v1 columns match the stored v2 file exactly.")
else:
    for c in V2_NEW:
        df[c] = v2_cols[c]
    print("Loaded v1, so the 6 v2 columns were added in memory (not saved).")

assert not (df["is_rush_hour"].astype(bool) & df["is_night"].astype(bool)).any(), "rush and night overlap"
display(df[V2_NEW].describe().T[["mean", "min", "50%", "max"]].round(4))

In [ ]:
# Holiday check against the meter (Vendor 2 only, because Vendor 1's `extra` also bundles other fees; see Section 7)
HOLIDAYS = pd.to_datetime(["2026-05-25", "2026-06-19"])   # Memorial Day (Mon), Juneteenth (Fri)
v2v = df[df["VendorID"] == 2]
is_hol = v2v["tpep_pickup_datetime"].dt.normalize().isin(HOLIDAYS)
win = v2v["tpep_pickup_datetime"].dt.hour.between(16, 19) & (v2v["tpep_pickup_datetime"].dt.dayofweek <= 4)
print(f"Vendor 2, weekday 16-19h, share charged extra >= 2.50:  ordinary weekdays {(v2v.loc[win & ~is_hol, 'extra'] >= 2.5).mean():.1%}"
      f"  |  the two holidays {(v2v.loc[win & is_hol, 'extra'] >= 2.5).mean():.1%}  ({int((win & is_hol).sum())} trips)")

**Conclusion.** The six v2 columns are deterministic functions of v1 columns (no randomness, no statistics learned from
data), so adding them before the split cannot leak. `is_rush_hour` ≈ 21% and `is_night` ≈ 28% of trips; `high_tip_bill`
≈ 16% of trips, because about half of all riders tip exactly the 20% preset.

## 7. Money-field check: `extra` and the bill

**Problem.** If the fare components are added up, they should equal `total_amount − tip_amount`. For Vendor 2 they do; for
Vendor 1 (Creative Mobile, ~18% of trips) they almost never do.

**Finding.** Vendor 1's `extra` already **includes** the congestion surcharge ($2.50), the CBD fee ($0.75), and the airport
fee, which also appear in their own columns. Summing everything therefore double-counts those fees for Vendor 1.

**Decision.** No rows or values change (no cleaning rule used `extra`). The bill is always computed as
`bill = total_amount − tip_amount`, which needs no component sum. If `extra` is used as a model feature, Vendor 1's
value must first be reduced by those three fees so both vendors mean the same thing.

In [ ]:
pre_tip = df["total_amount"] - df["tip_amount"]
base = df[["fare_amount", "extra", "mta_tax", "tolls_amount", "improvement_surcharge"]].sum(axis=1)
fees = df[["congestion_surcharge", "cbd_congestion_fee", "Airport_fee"]].fillna(0).sum(axis=1)
reconciles = lambda s: (s - pre_tip).abs() <= 0.011

recon = pd.DataFrame({
    "trips": df["VendorID"].value_counts(),
    "sum incl. 3 fee columns = bill": reconciles(base + fees).groupby(df["VendorID"]).mean(),
    "sum excl. 3 fee columns = bill": reconciles(base).groupby(df["VendorID"]).mean(),
}).sort_index()
display(recon.style.format({"trips": "{:,}", "sum incl. 3 fee columns = bill": "{:.1%}", "sum excl. 3 fee columns = bill": "{:.1%}"}))

print("Most common `extra` values, Vendor 1 (3.25 = 2.50 + 0.75; 5.75 = + 2.50 rush):")
print(df.loc[df["VendorID"] == 1, "extra"].round(2).value_counts().head(6).to_string())
print("Trips with bill < fare:", int((pre_tip.round(2) < df["fare_amount"]).sum()))

**Conclusion.** Vendor 1 (145,315 trips): ~1.4% reconcile with all fee columns, 100% once the three bundled fees are left
out; Vendor 2 (651,674 trips): 99.9% reconcile with all columns. The money fields are raw TLC values, and
`total_amount − tip_amount` is a consistent pre-tip bill for both vendors.

## 8. Time-based split

| Fold | Pickup dates | Days | Used for |
| --- | --- | --- | --- |
| train | Apr 1 – May 15, 2026 | 45 | fitting models, imputers, scalers, encoders |
| validation | May 16 – May 31, 2026 | 16 | tuning and model choice |
| test | June 1 – June 30, 2026 | 30 | final evaluation, touched once |

**Why by time and not at random.** Trips on the same day share weather, events, and traffic, so a random split would put
near-copies of test days into training and overstate performance. A time split also matches use: learn from the past,
evaluate on a later month. `pickup_date` is kept as the cluster unit for date-clustered standard errors and the day-block
bootstrap. (The full leakage discussion is in the modeling part.)

In [ ]:
split_check = np.select([df["tpep_pickup_datetime"] < TRAIN_END, df["tpep_pickup_datetime"] < VAL_END],
                        ["train", "val"], default="test")
assert (df["split"].to_numpy() == split_check).all(), "stored split does not follow the date rule"
assert (df["high_tip"] == (df["tip_amount"] / df["fare_amount"] >= HIGH_TIP_THRESHOLD)).all()

split_summary = (df.groupby("split")
                   .agg(rows=("high_tip", "size"), days=("pickup_date", "nunique"),
                        first_day=("pickup_date", "min"), last_day=("pickup_date", "max"),
                        high_tip_rate=("high_tip", "mean"), high_tip_bill_rate=("high_tip_bill", "mean"))
                   .reindex(["train", "val", "test"]))
split_summary["share_of_rows"] = split_summary["rows"] / split_summary["rows"].sum()
display(split_summary.style.format({"rows": "{:,}", "high_tip_rate": "{:.1%}", "high_tip_bill_rate": "{:.1%}", "share_of_rows": "{:.1%}"}))

assert split_summary["rows"].to_dict() == EXPECTED_SPLIT_ROWS, split_summary["rows"].to_dict()
assert split_summary["high_tip_rate"].round(3).to_dict() == EXPECTED_SPLIT_HIGH_TIP, split_summary["high_tip_rate"].round(3).to_dict()
assert split_summary["days"].tolist() == [45, 16, 30]
print("Split counts, days, and high-tip rates match the fixed values.")

**Conclusion.** Train 395,144 / validation 127,349 / test 274,496 trips (49.6% / 16.0% / 34.4%). The high-tip rate is
63.3% / 62.4% / 63.0%, so the folds are comparable and no fold is starved (the reservoir bug would have left validation
with ~11k trips).

## 9. Output, data dictionary, and summary

**Output file** (built earlier, read here): `MyDrive/CIS5450_project/data/clean/trips_clean_v2.parquet`, 796,989 rows × 54
columns. `trips_clean_v1.parquet` (first 48 columns) is kept unchanged. Parquet is the source of truth because it keeps
column types; the CSV copies of v1 are for viewing only.

**Feature column:** ✅ usable as a feature · ⚠️ usable after a train-only fix · ❌ never a feature.

In [ ]:
DICTIONARY = [
    # column, meaning, source, feature
    ("VendorID", "meter vendor: 1 = Creative Mobile, 2 = Curb", "TLC", "✅ categorical"),
    ("tpep_pickup_datetime", "pickup time (NYC local)", "TLC", "❌ use derived columns"),
    ("tpep_dropoff_datetime", "drop-off time", "TLC", "❌ use duration_min"),
    ("passenger_count", "passengers; 0 set to missing (3,047 trips)", "TLC", "⚠️ impute on train"),
    ("trip_distance", "miles, 0-100", "TLC", "✅"),
    ("RatecodeID", "1 standard, 2 JFK, 3 Newark, 4 Nassau/Westchester, 5 negotiated", "TLC", "✅ categorical"),
    ("store_and_fwd_flag", "Y = trip stored in the vehicle before upload", "TLC", "✅ (little information)"),
    ("PULocationID", "pickup taxi zone 1-263", "TLC", "✅ categorical"),
    ("DOLocationID", "drop-off taxi zone 1-263", "TLC", "✅ categorical"),
    ("payment_type", "always 1 (card)", "TLC", "❌ constant"),
    ("fare_amount", "time-and-distance fare, > 0; denominator of tip_pct", "TLC", "✅"),
    ("extra", "rush $2.50 / overnight $1; Vendor 1 also bundles congestion, CBD, airport fees", "TLC", "⚠️ de-duplicate Vendor 1"),
    ("mta_tax", "MTA tax, almost always $0.50", "TLC", "✅"),
    ("tip_amount", "card tip, > 0 and <= fare", "TLC", "❌ outcome"),
    ("tolls_amount", "tolls", "TLC", "✅"),
    ("improvement_surcharge", "improvement surcharge, almost always $1", "TLC", "✅"),
    ("total_amount", "total charged, INCLUDES the tip", "TLC", "❌ contains outcome"),
    ("congestion_surcharge", "NY State congestion surcharge, $0 or $2.50", "TLC", "✅"),
    ("Airport_fee", "JFK / LGA pickup fee", "TLC", "✅"),
    ("cbd_congestion_fee", "Manhattan CBD congestion fee, $0 or $0.75", "TLC", "✅ (or has_cbd_fee)"),
    ("source_month", "month file the trip was sampled from", "derived", "❌ bookkeeping"),
    ("duration_min", "drop-off minus pickup, 1-180 min", "derived", "✅"),
    ("PU_Borough", "pickup borough", "zone join", "✅ categorical"),
    ("PU_Zone", "pickup zone name", "zone join", "⚠️ duplicates PULocationID"),
    ("PU_service_zone", "Yellow Zone / Airports / Boro Zone / EWR", "zone join", "✅ categorical"),
    ("DO_Borough", "drop-off borough", "zone join", "✅ categorical"),
    ("DO_Zone", "drop-off zone name", "zone join", "⚠️ duplicates DOLocationID"),
    ("DO_service_zone", "as PU_service_zone, drop-off", "zone join", "✅ categorical"),
    ("pickup_hour_ts", "pickup time floored to the hour (weather join key)", "derived", "❌ join key"),
    ("temp_c", "temperature in the pickup hour (°C)", "weather join", "✅"),
    ("precip_mm", "precipitation in the hour ending at pickup_hour_ts (mm)", "weather join", "✅"),
    ("is_rain", "precip_mm >= 0.1", "derived", "✅"),
    ("pickup_hour", "pickup hour 0-23", "derived", "✅"),
    ("pickup_dow", "day of week, 0 = Monday", "derived", "✅"),
    ("is_weekend", "Saturday or Sunday", "derived", "✅"),
    ("is_late_night", "pickup 00:00-05:59 (subset of is_night)", "derived", "✅"),
    ("avg_speed_mph", "60 x distance / duration", "derived", "⚠️ cap chosen on train"),
    ("is_airport", "pickup or drop-off at JFK (132) / LGA (138)", "derived", "✅"),
    ("is_newark", "pickup or drop-off at Newark (1)", "derived", "✅"),
    ("same_borough", "pickup borough == drop-off borough", "derived", "✅"),
    ("has_cbd_fee", "cbd_congestion_fee > 0", "derived", "✅"),
    ("is_short_trip", "trip_distance <= 2 mi", "derived", "✅"),
    ("tip_pct", "tip_amount / fare_amount, <= 1", "derived", "❌ outcome"),
    ("high_tip", "tip_pct >= 0.25 (proposal target)", "derived", "❌ target"),
    ("high_tip_20", "tip_pct >= 0.20 (sensitivity)", "derived", "❌ target"),
    ("high_tip_30", "tip_pct >= 0.30 (sensitivity)", "derived", "❌ target"),
    ("pickup_date", "pickup date (cluster unit for SEs / bootstrap)", "derived", "❌ bookkeeping"),
    ("split", "train / val / test by pickup date", "derived", "❌ bookkeeping"),
    ("is_rush_hour", "Mon-Fri pickup 16:00-19:59, holidays included", "v2", "✅"),
    ("is_night", "pickup 20:00-05:59", "v2", "✅"),
    ("tip_pct_bill", "tip / (total_amount - tip_amount)", "v2", "❌ outcome"),
    ("high_tip_bill", "tip > round(0.20 x bill, 2) + 0.015", "v2", "❌ target"),
    ("high_tip_bill_25", "same rule at 25%", "v2", "❌ target"),
    ("high_tip_bill_30", "same rule at 30%", "v2", "❌ target"),
]
dictionary = pd.DataFrame(DICTIONARY, columns=["column", "meaning", "source", "feature"]).set_index("column")
undocumented = [c for c in df.columns if c not in dictionary.index]
if undocumented:
    print("WARNING: columns without a dictionary entry:", undocumented)
dictionary = dictionary.reindex(df.columns).fillna("?")
dictionary.insert(0, "dtype", df.dtypes.astype(str))
dictionary.insert(1, "missing", df.isna().sum())
dictionary = dictionary.reset_index().rename(columns={"index": "column"})
dictionary.index = np.arange(1, len(dictionary) + 1)
display(dictionary)

NOT_FEATURES = dictionary.loc[dictionary["feature"].str.startswith("❌"), "column"].tolist()
print(f"{len(NOT_FEATURES)} columns must never be model features:", NOT_FEATURES)

In [ ]:
# Final integrity check: the table this notebook loaded is unchanged, and no file was written.
assert len(df) == EXPECTED_ROWS
assert pd.util.hash_pandas_object(df[LOADED_COLS], index=True).sum() == LOADED_FINGERPRINT, "the loaded data changed"
print(f"OK: {len(df):,} rows; the loaded {LOADED_FROM.name} columns are unchanged; this notebook wrote no files.")
con.close()

### Summary

- **Data.** NYC TLC yellow taxi records for April–June 2026 (11,759,324 trips; 7,796,205 card trips), the Taxi Zone
  Lookup Table, and Open-Meteo hourly weather. Downloaded once on 2026-11-03 and kept on Drive.
- **Sampling.** DuckDB, card trips only, 300,000 per month with `SEED = 5450` → 900,000 trips. Hash-based sampling replaced
  reservoir sampling after the reservoir sample proved time-biased (validation fold of 10,836 trips); every day's share in
  the final sample is within 0.07 pp of the full month.
- **Cleaning.** Eight rules, each with a reason and a count: 900,000 → 825,233 (positive tip and fare) → **796,989** trips
  (88.6%). Zero tips (74,765) and implausible durations (20,509) account for most removals. Slow trips are kept on purpose;
  `passenger_count = 0` becomes missing rather than dropping the trip.
- **Joins.** Zone (pickup + drop-off) and hourly weather LEFT JOINs both match 100% of trips with no row-count change;
  weather is aligned to the last completed hour before pickup. 16.2% of trips are in rainy hours.
- **v2 features.** `is_rush_hour` (holidays included, as the meters charged the surcharge), `is_night`, and the bill-based
  outcome `tip_pct_bill` / `high_tip_bill` (+25%, 30%). All deterministic, all verified here.
- **Money fields.** Raw TLC values, unchanged. Vendor 1's `extra` bundles three fees, so the bill is `total − tip` and
  `extra` must be de-duplicated before use as a feature.
- **Split.** By pickup date: train 395,144 / validation 127,349 / test 274,496; high-tip rate 63.3% / 62.4% / 63.0%.
- **Leakage guard.** `tip_amount`, `total_amount`, and everything derived from them are never features; nothing in this
  part learns a statistic from the data, so imputation, capping, encoding, and scaling are fitted on train in the modeling part.

**Limitations.** Trips with a zero card tip are excluded, so results describe riders who tipped by card, not all riders.
91% of pickups are in Manhattan, so outer-borough patterns are imprecise. Weather is a single Manhattan point.
Early weekday mornings (Mon–Wed, ~2–5 AM) have only ~150–330 sampled trips per hour-by-day cell.